In [1]:
%pip install opencv-python numpy

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


Εγκαθιστούμε τις βασικές βιβλιοθήκες που χρειάζεται η εφαρμογή. Η OpenCV χρησιμοποιείται για την κάμερα, την επεξεργασία εικόνας, τα φίλτρα ακμών και τα contours. Η NumPy χρησιμοποιείται για αριθμητικές πράξεις πάνω στις εικόνες.

In [2]:
import cv2
import numpy as np
import time

Το `cv2` είναι η `OpenCV`. Το `numpy` χρησιμοποιείται για πράξεις πάνω σε πίνακες εικόνων. Το `time` χρησιμοποιείται για τον υπολογισμό των FPS, δηλαδή των καρέ ανά δευτερόλεπτο.

In [3]:
CAMERA_INDEX = 0
FRAME_WIDTH = 800

MIN_CONTOUR_AREA = 2500
MIN_BOX_AREA = 5000
MAX_OBJECTS_TO_DRAW = 5

Το `CAMERA_INDEX = 0` σημαίνει ότι χρησιμοποιείται η βασική κάμερα του υπολογιστή.
Το `FRAME_WIDTH = 800` κάνει resize το βίντεο για να τρέχει πιο γρήγορα η εφαρμογή.
Τα `MIN_CONTOUR_AREA` και `MIN_BOX_AREA` χρησιμοποιούνται για να αγνοούμε πολύ μικρά contours, τα οποία συνήθως είναι θόρυβος.
Το `MAX_OBJECTS_TO_DRAW` περιορίζει πόσα αντικείμενα θα σχεδιάζονται κάθε φορά.

In [4]:
def resize_frame(frame, width=FRAME_WIDTH):
    height, original_width = frame.shape[:2]

    scale = width / original_width
    new_height = int(height * scale)

    resized = cv2.resize(frame, (width, new_height))

    return resized

Η κάμερα μπορεί να επιστρέφει εικόνα σε μεγάλο μέγεθος. Για να είναι πιο γρήγορη η εφαρμογή, κάνουμε resize κάθε frame σε σταθερό πλάτος. Κρατάμε όμως το σωστό aspect ratio, ώστε η εικόνα να μη φαίνεται παραμορφωμένη.

In [5]:
def preprocess_frame(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    return blurred

Πρώτα μετατρέπουμε την εικόνα σε grayscale, γιατί οι μέθοδοι ανίχνευσης ακμών βασίζονται κυρίως στις μεταβολές φωτεινότητας.
Μετά εφαρμόζουμε Gaussian Blur για να μειωθεί ο θόρυβος. Αυτό βοηθάει ώστε να μη δημιουργούνται πολλές ψεύτικες ακμές.

In [6]:
def normalize_to_uint8(image):
    normalized = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)
    normalized = normalized.astype(np.uint8)

    return normalized

Οι μέθοδοι `Sobel` και `Laplacian` μπορεί να παράγουν τιμές εκτός του εύρους 0–255. Η συνάρτηση αυτή μετατρέπει το αποτέλεσμα σε εικόνα τύπου `uint8`, ώστε να μπορεί να εμφανιστεί σωστά και να χρησιμοποιηθεί για thresholding.

In [7]:
def apply_edge_detection(gray, method):
    if method == "canny":
        edges = cv2.Canny(gray, 50, 150)
        return edges

    elif method == "sobel":
        grad_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        grad_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

        magnitude = cv2.magnitude(grad_x, grad_y)
        magnitude = normalize_to_uint8(magnitude)

        _, edges = cv2.threshold(
            magnitude,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        return edges

    elif method == "laplacian":
        lap = cv2.Laplacian(gray, cv2.CV_64F, ksize=3)
        lap = np.absolute(lap)
        lap = normalize_to_uint8(lap)

        _, edges = cv2.threshold(
            lap,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )

        return edges

    else:
        raise ValueError("Άγνωστη μέθοδος edge detection")

Η συνάρτηση αυτή εφαρμόζει μία από τις τρεις μεθόδους ανίχνευσης ακμών.

Ο `Canny` παράγει συνήθως λεπτές και καθαρές ακμές.
Ο `Sobel` υπολογίζει τις μεταβολές φωτεινότητας στους άξονες x και y.
Ο `Laplacian` βασίζεται στη δεύτερη παράγωγο και μπορεί να εντοπίζει ακμές προς πολλές κατευθύνσεις, αλλά είναι πιο ευαίσθητος στον θόρυβο.

In [8]:
def clean_edges(edges):
    kernel = np.ones((5, 5), np.uint8)

    closed = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel, iterations=2)

    dilated = cv2.dilate(closed, kernel, iterations=1)

    return dilated

Μετά την ανίχνευση ακμών, οι γραμμές μπορεί να έχουν μικρά κενά.
Το `MORPH_CLOSE` βοηθάει να κλείσουν αυτά τα κενά.
Το `dilate` παχαίνει λίγο τις ακμές, ώστε τα contours να βγαίνουν πιο ενωμένα και πιο σταθερά.

In [9]:
def classify_object(contour, frame_area):
    area = cv2.contourArea(contour)

    if area < MIN_CONTOUR_AREA:
        return None

    x, y, w, h = cv2.boundingRect(contour)
    box_area = w * h

    if box_area < MIN_BOX_AREA:
        return None

    if box_area > 0.45 * frame_area:
        return None

    if w < 40 or h < 40:
        return None

    aspect_ratio = w / float(h)
    long_ratio = max(w, h) / float(min(w, h))
    extent = area / float(box_area)

    perimeter = cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, 0.04 * perimeter, True)
    vertices = len(approx)

    # 1. Sunglasses: φαρδύ και χαμηλό αντικείμενο
    if (
        aspect_ratio >= 2.2
        and 0.10 <= extent <= 0.75
        and 4 <= vertices <= 25
    ):
        label = "Sunglasses"
        color = (255, 0, 255)

    # 2. Book / Notebook: μεγάλο ορθογώνιο αντικείμενο
    elif (
        1.35 <= long_ratio <= 3.5
        and extent >= 0.45
        and 4 <= vertices <= 14
    ):
        label = "Book/Notebook"
        color = (255, 0, 0)

    # 3. Box: πιο τετράγωνο και συμπαγές αντικείμενο
    elif (
        0.75 <= aspect_ratio <= 1.35
        and extent >= 0.45
        and 4 <= vertices <= 14
    ):
        label = "Box"
        color = (0, 165, 255)

    else:
        return None

    features = {
        "x": x,
        "y": y,
        "w": w,
        "h": h,
        "area": area,
        "box_area": box_area,
        "aspect_ratio": aspect_ratio,
        "extent": extent,
        "vertices": vertices
    }

    return label, color, features

Εδώ γίνεται η απλή αναγνώριση των 3 αντικειμένων. Δεν χρησιμοποιείται τεχνητή νοημοσύνη ή εκπαιδευμένο μοντέλο. Η κατηγοριοποίηση γίνεται με απλούς κανόνες πάνω στα γεωμετρικά χαρακτηριστικά των contours.

Τα γυαλιά θεωρούνται φαρδύ και χαμηλό αντικείμενο.
Το βιβλίο/τετράδιο θεωρείται ορθογώνιο αντικείμενο με μία πλευρά αρκετά μεγαλύτερη από την άλλη.
Το κουτί θεωρείται πιο τετράγωνο και συμπαγές αντικείμενο.

In [10]:
def draw_result(frame, contour, label, color, features):
    x = features["x"]
    y = features["y"]
    w = features["w"]
    h = features["h"]

    cv2.drawContours(frame, [contour], -1, color, 2)

    cv2.rectangle(
        frame,
        (x, y),
        (x + w, y + h),
        color,
        2
    )

    cv2.putText(
        frame,
        label,
        (x, max(y - 10, 25)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        color,
        2
    )

Για κάθε αντικείμενο που αναγνωρίζεται, σχεδιάζεται το contour, το bounding box και η ετικέτα του αντικειμένου.
Χρησιμοποιούμε διαφορετικό χρώμα για κάθε κατηγορία, ώστε να ξεχωρίζουν οπτικά τα αποτελέσματα.

In [11]:
def run_realtime_app():
    cap = cv2.VideoCapture(CAMERA_INDEX)

    if not cap.isOpened():
        print("Δεν μπόρεσε να ανοίξει η κάμερα.")
        return

    method = "canny"
    previous_time = time.time()

    window_name = "Real-Time Edge Detection and Object Contours"
    edges_window_name = "Edges"

    print("Η εφαρμογή ξεκίνησε.")
    print("Πάτησε 1 για Canny.")
    print("Πάτησε 2 για Sobel.")
    print("Πάτησε 3 για Laplacian.")
    print("Πάτησε q ή ESC για έξοδο.")

    try:
        while True:
            ret, frame = cap.read()

            if not ret:
                print("Δεν μπόρεσε να διαβαστεί frame από την κάμερα.")
                break

            frame = resize_frame(frame)
            output = frame.copy()

            gray = preprocess_frame(frame)

            edges = apply_edge_detection(gray, method)
            cleaned_edges = clean_edges(edges)

            contours, _ = cv2.findContours(
                cleaned_edges,
                cv2.RETR_EXTERNAL,
                cv2.CHAIN_APPROX_SIMPLE
            )

            frame_area = frame.shape[0] * frame.shape[1]

            detected_objects = []

            for contour in contours:
                result = classify_object(contour, frame_area)

                if result is not None:
                    label, color, features = result
                    detected_objects.append((contour, label, color, features))

            detected_objects = sorted(
                detected_objects,
                key=lambda item: item[3]["box_area"],
                reverse=True
            )

            for contour, label, color, features in detected_objects[:MAX_OBJECTS_TO_DRAW]:
                draw_result(output, contour, label, color, features)

            current_time = time.time()
            fps = 1 / (current_time - previous_time)
            previous_time = current_time

            cv2.putText(
                output,
                f"Method: {method.upper()} | FPS: {fps:.1f} | Objects: {len(detected_objects)}",
                (15, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 255),
                2
            )

            cv2.putText(
                output,
                "Keys: 1=Canny, 2=Sobel, 3=Laplacian, q/ESC=Exit",
                (15, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                (0, 255, 255),
                2
            )

            cv2.imshow(window_name, output)
            cv2.imshow(edges_window_name, cleaned_edges)

            key = cv2.waitKey(1) & 0xFF

            if key == ord("1"):
                method = "canny"
                print("Μέθοδος: Canny")

            elif key == ord("2"):
                method = "sobel"
                print("Μέθοδος: Sobel")

            elif key == ord("3"):
                method = "laplacian"
                print("Μέθοδος: Laplacian")

            elif key == ord("q") or key == 27:
                print("Τερματισμός εφαρμογής.")
                break

            if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
                print("Το παράθυρο έκλεισε.")
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()

        for _ in range(5):
            cv2.waitKey(1)

        print("Η κάμερα απελευθερώθηκε και τα παράθυρα έκλεισαν.")

Αυτή είναι η βασική συνάρτηση της εφαρμογής. Ανοίγει την κάμερα, διαβάζει frames σε πραγματικό χρόνο, εφαρμόζει την επιλεγμένη μέθοδο ανίχνευσης ακμών, βρίσκει contours και προσπαθεί να αναγνωρίσει τα τρία αντικείμενα.

Με τα πλήκτρα:

1-> Canny

2 -> Sobel

3 -> Laplacian

q ή ESC για έξοδο

Στην οθόνη εμφανίζονται η ενεργή μέθοδος, τα FPS και ο αριθμός των αντικειμένων που εντοπίστηκαν.